In [5]:
import math
import pandas as pd


data = {
    'Outlook': ['Sunny','Sunny','Overcast','Rain','Rain','Rain',
                'Overcast','Sunny','Sunny','Rain','Sunny','Overcast',
                'Overcast','Rain'],

    'Temperature': ['Hot','Hot','Hot','Mild','Cool','Cool',
                    'Cool','Mild','Cool','Mild','Mild','Mild',
                    'Hot','Mild'],

    'Humidity': ['High','High','High','High','Normal','Normal',
                 'Normal','High','Normal','Normal','Normal','High',
                 'Normal','High'],

    'Wind': ['Weak','Strong','Weak','Weak','Weak','Strong',
             'Strong','Weak','Weak','Weak','Strong','Strong',
             'Weak','Strong'],

    'PlayTennis': ['No','No','Yes','Yes','Yes','No',
                   'Yes','No','Yes','Yes','Yes','Yes',
                   'Yes','No']
}

df = pd.DataFrame(data)


def entropy(column):
    values = column.value_counts()
    total = len(column)

    result = 0

    for count in values:
        p = count / total
        result -= p * math.log2(p)

    return result


def information_gain(df, attribute, target):
    total_entropy = entropy(df[target])

    values = df[attribute].unique()
    weighted_entropy = 0

    for value in values:
        subset = df[df[attribute] == value]
        weighted_entropy += (len(subset) / len(df)) * entropy(subset[target])

    return total_entropy - weighted_entropy


def ID3(df, target, attributes):

    # If all examples have same target
    if len(df[target].unique()) == 1:
        return df[target].iloc[0]

    # If no attributes remain
    if len(attributes) == 0:
        return df[target].mode()[0]

    # Find attribute with highest information gain
    gains = []

    for attribute in attributes:
        gains.append(information_gain(df, attribute, target))

    best_attribute = attributes[gains.index(max(gains))]

    tree = {best_attribute: {}}

    remaining = [a for a in attributes if a != best_attribute]

    for value in df[best_attribute].unique():

        subset = df[df[best_attribute] == value]

        tree[best_attribute][value] = ID3(
            subset,
            target,
            remaining
        )

    return tree


attributes = ['Outlook', 'Temperature', 'Humidity', 'Wind']

tree = ID3(df, 'PlayTennis', attributes)

print("Decision Tree:")
print(tree)


def predict(tree, sample):

    if not isinstance(tree, dict):
        return tree

    attribute = next(iter(tree))
    value = sample[attribute]

    return predict(tree[attribute][value], sample)


sample = {
    'Outlook': 'Sunny',
    'Temperature': 'Cool',
    'Humidity': 'High',
    'Wind': 'Strong'
}

result = predict(tree, sample)

print("\nNew Sample:")
print(sample)

print("\nPrediction:", result)

Decision Tree:
{'Outlook': {'Sunny': {'Humidity': {'High': 'No', 'Normal': 'Yes'}}, 'Overcast': 'Yes', 'Rain': {'Wind': {'Weak': 'Yes', 'Strong': 'No'}}}}

New Sample:
{'Outlook': 'Sunny', 'Temperature': 'Cool', 'Humidity': 'High', 'Wind': 'Strong'}

Prediction: No
